# Cleaning the Amozon Scrap Data

In [32]:
import numpy as np
import pandas as pd


In [33]:
# step 1: Import the Data
df=pd.read_csv(r"C:\Data Science\Project\Web_Scraping_Project\amazon_laptop_scrap.csv")

In [30]:
# step 2: Understand the Data
df.head()

df.sample(5)

df.columns

Index(['Brand', 'Price', 'Rating', 'Ram', 'Color', 'Processor', 'Storage_Gb'], dtype='str')

In [4]:
# step 3: Check shape and Size of Data
df.dtypes

df.shape

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1129 entries, 0 to 1128
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      1129 non-null   str    
 1   price      1125 non-null   str    
 2   rating     1070 non-null   float64
 3   brand      1129 non-null   str    
 4   ram        1098 non-null   str    
 5   storage    1091 non-null   str    
 6   color      901 non-null    str    
 7   processor  1099 non-null   str    
dtypes: float64(1), str(7)
memory usage: 70.7 KB


In [5]:
# step 4: Find the Missing Value
df.isnull().sum()

title          0
price          4
rating        59
brand          0
ram           31
storage       38
color        228
processor     30
dtype: int64

In [6]:
# step 5 : find the duplicated
df["title"].duplicated().sum()

np.int64(846)

In [7]:
# step 6: Clean the Data by ONE by ONE colume
df.dtypes



title            str
price            str
rating       float64
brand            str
ram              str
storage          str
color            str
processor        str
dtype: object

In [8]:
df["price"]=df["price"].str.replace(",","")

In [9]:
# Step 7: Clean Price and Rating

df[df["price"].isna()]      # check the null value in price colume
df[df["rating"].isna()]     # check the null value in rating colume 


df["price"] = pd.to_numeric(df["price"], errors="coerce")      # Convert to numeric
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")    # Convert to numeric


df.dropna(subset=["price", "rating"], inplace=True)                   # Drop rows where price or rating is missing

print("Null in Price:", df["price"].isna().sum())
print("Null in Rating:", df["rating"].isna().sum())

print("*" * 100)

df.head()

Null in Price: 0
Null in Rating: 0
****************************************************************************************************


,title,price,rating,brand,ram,storage,color,processor
0,"HP Omnibook 3, Snapdragon X Processor 45 Tops ...",69990.0,3.4,HP,16GB,512GB,Silver,Apple M365
1,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,73990.0,4.8,Apple,8GB,256GB,NaN,Apple A18
2,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,44999.0,4.0,Lenovo,8GB,512GB,Silver,AMD
3,"ASUS Vivobook 15, Smartchoice,Intel Core i5 13...",65990.0,4.1,ASUS,16GB,512GB,Blue,Apple M365
4,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,73990.0,4.7,Apple,8GB,256GB,Silver,Apple A18


In [10]:
df.dtypes
#df.head()
df["rating"].unique()

array([3.4, 4.8, 4. , 4.1, 4.7, 3.8, 5. , 3.6, 4.2, 1. , 3.9, 2.8, 4.6,
       4.3, 3.7, 4.4, 4.5, 3.2, 3.3, 3. , 3.5, 2. , 1.6, 2.9, 2.4, 3.1,
       4.9, 2.6])

In [11]:
# step 8 : cleaning colume brand
df["brand"] = df["brand"].str.strip().str.title().replace({"Hp":"HP","Asus":"ASUS"})

df = df[~df["brand"].isin(["Certified", "Maxbook", "Msi", "New"])]    # drop row of this brand ["Certified", "Maxbook", "Msi", "New"] 
df["brand"].head()

0        HP
1     Apple
2    Lenovo
3      ASUS
4     Apple
Name: brand, dtype: str

In [12]:
df["brand"].value_counts()

brand
Apple        587
HP           234
Lenovo       139
Acer          39
ASUS          27
Dell          20
Samsung        4
Primebook      2
Alienware      2
Microsoft      2
Name: count, dtype: int64

In [13]:
# step 9 : Cleaning Colume Ram
print(df["ram"].unique())
df=df[~df["ram"].isin(["RAM","DDR5","LPDDR4","DDR7"])]    # drop this Ram_types

df["ram"].value_counts()

<StringArray>
[  '16GB',    '8GB',   '36GB',    'RAM', 'LPDDR4',   '12GB',   'DDR5',
   '24GB',    '4GB',    '6GB',      nan,   '32GB',  '128GB',   'DDR7',
   '48GB']
Length: 15, dtype: str


ram
16GB     421
8GB      284
24GB     146
36GB     100
48GB      39
4GB        7
6GB        7
32GB       7
12GB       2
128GB      2
Name: count, dtype: int64

In [14]:
df["ram"] = df["ram"].str.extract(r"(\d+)")[0]          # remove Gb
df["ram"] =pd.to_numeric(df["ram"], errors="coerce")    # convert to Numeric

In [15]:
df=df.dropna(subset=["ram"])     # remove nan
print(df["ram"].unique())

[ 16.   8.  36.  12.  24.   4.   6.  32. 128.  48.]


In [16]:
# step 9 : Cleaning Colume Ram
df["storage"].unique()
df["storage_value"] = df["storage"].str.extract(r"(\d+\.?\d*)")[0].astype(float)
df["storage_gb"] = df["storage_value"]
df.loc[df["storage"].str.contains("TB", case=False, na=False), "storage_gb"] *= 1000
df["storage_gb"] =pd.to_numeric(df["storage_gb"], errors="coerce")
df=df.dropna(subset=["storage_gb"])
df["storage_gb"].unique()

array([ 512.,  256., 1000., 2000.,   64.,  128.,  500.])

In [17]:
# step 10 : Clean Color Colume 
import random

colors = ["Black", "Silver", "Gray", "White"]

df["color"] = df["color"].str.strip().str.title()

mask = df["color"].isna()

df.loc[mask, "color"] = [
    random.choice(colors)
    for _ in range(mask.sum())
]

In [18]:
df["color"] = df["color"].str.strip().str.title()
df["color"]=df["color"].str.replace("Grey","Gray")
df["color"].unique()

<StringArray>
['Silver', 'White', 'Blue', 'Gray', 'Black', 'Indigo']
Length: 6, dtype: str

In [19]:
# step 11 : Clean Processor Columns

df.dropna(subset=["processor"],inplace=True)
df["processor"].isna().sum()

np.int64(0)

In [20]:
# Step 12 : Drop Extra Columns

df.drop(["storage","storage_value","title"],axis=1,inplace =True)

In [21]:
df.columns

Index(['price', 'rating', 'brand', 'ram', 'color', 'processor', 'storage_gb'], dtype='str')

In [22]:
# Step 13 : Set Brand as First Column

#brand = df.pop("brand")
#df.insert(0, "brand", brand)

# Get columns as a list
cols = list(df.columns)

# Pop the column you want to move and insert it at index 0
cols.insert(0, cols.pop(cols.index('brand')))

# Apply the new order
df = df[cols]


In [23]:
df["brand"].unique()

<StringArray>
[       'HP',     'Apple',    'Lenovo',      'ASUS',      'Dell',      'Acer',
 'Alienware',   'Samsung', 'Microsoft']
Length: 9, dtype: str

In [24]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 792


In [25]:
# mathematics 
df.describe()

,price,rating,ram,storage_gb
count,977.000000,977.00000,977.000000,977.000000
mean,180840.297851,4.27697,18.483112,863.856704
std,160074.385456,0.70742,11.485735,593.980512
min,24990.000000,1.00000,4.000000,64.000000
25%,70990.000000,3.90000,8.000000,512.000000
50%,83490.000000,4.60000,16.000000,512.000000
75%,200490.000000,4.80000,24.000000,1000.000000
max,999990.000000,5.00000,128.000000,2000.000000


In [26]:
# Check numerical columns
print(df[["price", "rating", "ram", "storage_gb"]].describe())

               price     rating         ram   storage_gb
count     977.000000  977.00000  977.000000   977.000000
mean   180840.297851    4.27697   18.483112   863.856704
std    160074.385456    0.70742   11.485735   593.980512
min     24990.000000    1.00000    4.000000    64.000000
25%     70990.000000    3.90000    8.000000   512.000000
50%     83490.000000    4.60000   16.000000   512.000000
75%    200490.000000    4.80000   24.000000  1000.000000
max    999990.000000    5.00000  128.000000  2000.000000


In [27]:
#  Check categorical values
print(df["brand"].unique())
print(df["color"].unique())
print(df["processor"].unique())

<StringArray>
[       'HP',     'Apple',    'Lenovo',      'ASUS',      'Dell',      'Acer',
 'Alienware',   'Samsung', 'Microsoft']
Length: 9, dtype: str
<StringArray>
['Silver', 'White', 'Blue', 'Gray', 'Black', 'Indigo']
Length: 6, dtype: str
<StringArray>
['Apple M365',  'Apple A18',        'AMD',   'Apple M5',      'Intel',
 'Apple A114',   'MediaTek', 'Snapdragon', 'Apple A325']
Length: 9, dtype: str


In [28]:
# improve colume Name

df.columns=df.columns.str.title()
df.head()

,Brand,Price,Rating,Ram,Color,Processor,Storage_Gb
0,HP,69990.0,3.4,16.0,Silver,Apple M365,512.0
1,Apple,73990.0,4.8,8.0,White,Apple A18,256.0
2,Lenovo,44999.0,4.0,8.0,Silver,AMD,512.0
3,ASUS,65990.0,4.1,16.0,Blue,Apple M365,512.0
4,Apple,73990.0,4.7,8.0,Silver,Apple A18,256.0


In [29]:
# Save the CLean DataSet
#df.to_csv("Clean_Dataset_Amazon.csv",index=False)

# MatplotLib 